In [1]:
import os
from pathlib import Path
import sys

root = Path(r"C:\Users\arius\Desktop\kalshi_wnba_bot")

os.chdir(root)  # <- THIS is the key line
sys.path.insert(0, str(root / "src"))
sys.path.insert(0, str(root / "notebooks"))

print("CWD now:", Path.cwd())
import srwnba
print(srwnba.__file__)
import pandas as pd
from gridsearch_elo import run_gridsearch

CWD now: C:\Users\arius\Desktop\kalshi_wnba_bot
C:\Users\arius\Desktop\kalshi_wnba_bot\src\srwnba\__init__.py


In [2]:
import importlib
import gridsearch_elo
importlib.reload(gridsearch_elo)

from gridsearch_elo import run_gridsearch

We must tune H, K, a, and b. H=Home Court Elo Advantage, K = Learning Rate, a =  Season to season ELO retention, b = MOV strength

We gridsearch all simultaneously for robustness.

In [ ]:
initial_train_years = [2015, 2016, 2017, 2018, 2019]
all_years = list(range(2015, 2025))

H_list = [35, 40, 43, 45, 50, 55]
K_list = [10, 15, 20, 25, 30]
a_list = [0.50, 0.60, 0.70, 0.75, 0.85]

# MOV choices:
# - no MOV  -> use_mov=False (b ignored, but we'll include b=0.8 as a placeholder)
# - MOV w/ exponent -> use_mov=True and b in {0.6,0.8,1.0}
b_list = [0, 0.6, 0.8, 1.0]

train = initial_train_years.copy()
summary_rows = []

while True:
    last_train = train[-1]
    idx = all_years.index(last_train)
    if idx + 1 >= len(all_years):
        break
    test_year = all_years[idx + 1]

    print(f"\n=== WALKFORWARD: train={train[0]}-{train[-1]} test={test_year} ===")

    res = run_gridsearch(
        train_years=train,
        test_years=[test_year],
        H_list=H_list,
        K_list=K_list,
        a_list=a_list,
        b_list=b_list,
        verbose=True,
    )

    best = res.iloc[0].to_dict()
    summary_rows.append({
        "train_start": train[0],
        "train_end": train[-1],
        "test_year": test_year,
        "best_H": best["H"],
        "best_K": best["K"],
        "best_a": best["a"],
        "best_b": best["b"],
        "train_logloss": best["train_logloss"],
        "test_logloss": best["test_logloss"],
        "train_brier": best["train_brier"],
        "test_brier": best["test_brier"],
    })

    train = train + [test_year]

summary = pd.DataFrame(summary_rows)
summary


=== WALKFORWARD: train=2015-2019 test=2020 ===
H=35 K=10 a=0.5 b=0 | train_logloss=0.647221 train_brier=0.227602 | test_logloss=0.641318 test_brier=0.224553
H=35 K=10 a=0.5 b=0.6 | train_logloss=0.651598 train_brier=0.229613 | test_logloss=0.650583 test_brier=0.229021
H=35 K=10 a=0.5 b=0.8 | train_logloss=0.639823 train_brier=0.224182 | test_logloss=0.631451 test_brier=0.219773
H=35 K=10 a=0.5 b=1 | train_logloss=0.629511 train_brier=0.219728 | test_logloss=0.612255 test_brier=0.210644
H=35 K=10 a=0.6 b=0 | train_logloss=0.646402 train_brier=0.227247 | test_logloss=0.636181 test_brier=0.222111
H=35 K=10 a=0.6 b=0.6 | train_logloss=0.650554 train_brier=0.229137 | test_logloss=0.645690 test_brier=0.226660
H=35 K=10 a=0.6 b=0.8 | train_logloss=0.638927 train_brier=0.223815 | test_logloss=0.626901 test_brier=0.217627
H=35 K=10 a=0.6 b=1 | train_logloss=0.629234 train_brier=0.219669 | test_logloss=0.609631 test_brier=0.209392
H=35 K=10 a=0.7 b=0 | train_logloss=0.645936 train_brier=0.22707

,train_start,train_end,test_year,best_H,best_K,best_a,best_b,train_logloss,test_logloss,train_brier,test_brier
0,2015,2019,2020,35,30,0.5,1.0,0.632631,0.591971,0.220706,0.199749
1,2015,2020,2021,35,30,0.6,0.8,0.623764,0.606534,0.217118,0.210775
2,2015,2021,2022,35,15,0.6,1.0,0.621091,0.610753,0.216107,0.211770
3,2015,2022,2023,35,20,0.5,1.0,0.619576,0.601675,0.215485,0.206994
4,2015,2023,2024,35,30,0.5,0.8,0.617772,0.596690,0.214619,0.203809


In [ ]:
initial_train_years = [2015, 2016, 2017, 2018, 2019]
all_years = list(range(2015, 2025))

H_list = [25, 30, 35, 40]
K_list = [15, 20, 25, 30, 35]
a_list = [0.45, 0.50, 0.55, 0.60, 0.65]
b_list = [0.8, 0.9, 1.0]

# MOV choices:
# - no MOV  -> use_mov=False (b ignored, but we'll include b=0.8 as a placeholder)
# - MOV w/ exponent -> use_mov=True and b in {0.6,0.8,1.0}

train = initial_train_years.copy()
summary_rows = []

while True:
    last_train = train[-1]
    idx = all_years.index(last_train)
    if idx + 1 >= len(all_years):
        break
    test_year = all_years[idx + 1]

    print(f"\n=== WALKFORWARD: train={train[0]}-{train[-1]} test={test_year} ===")

    res = run_gridsearch(
        train_years=train,
        test_years=[test_year],
        H_list=H_list,
        K_list=K_list,
        a_list=a_list,
        b_list=b_list,
        verbose=True,
    )

    best = res.iloc[0].to_dict()
    summary_rows.append({
        "train_start": train[0],
        "train_end": train[-1],
        "test_year": test_year,
        "best_H": best["H"],
        "best_K": best["K"],
        "best_a": best["a"],
        "best_b": best["b"],
        "train_logloss": best["train_logloss"],
        "test_logloss": best["test_logloss"],
        "train_brier": best["train_brier"],
        "test_brier": best["test_brier"],
    })

    train = train + [test_year]

summary = pd.DataFrame(summary_rows)
summary


=== WALKFORWARD: train=2015-2019 test=2020 ===
H=25 K=15 a=0.45 b=0.8 | train_logloss=0.635897 train_brier=0.222496 | test_logloss=0.618147 test_brier=0.213476
H=25 K=15 a=0.45 b=0.9 | train_logloss=0.631629 train_brier=0.220693 | test_logloss=0.608730 test_brier=0.209099
H=25 K=15 a=0.45 b=1 | train_logloss=0.629197 train_brier=0.219708 | test_logloss=0.600398 test_brier=0.205203
H=25 K=15 a=0.5 b=0.8 | train_logloss=0.635468 train_brier=0.222324 | test_logloss=0.616353 test_brier=0.212644
H=25 K=15 a=0.5 b=0.9 | train_logloss=0.631352 train_brier=0.220593 | test_logloss=0.607476 test_brier=0.208519
H=25 K=15 a=0.5 b=1 | train_logloss=0.629102 train_brier=0.219682 | test_logloss=0.599776 test_brier=0.204899
H=25 K=15 a=0.55 b=0.8 | train_logloss=0.635152 train_brier=0.222210 | test_logloss=0.614604 test_brier=0.211839
H=25 K=15 a=0.55 b=0.9 | train_logloss=0.631211 train_brier=0.220558 | test_logloss=0.606318 test_brier=0.207983
H=25 K=15 a=0.55 b=1 | train_logloss=0.629165 train_bri

,train_start,train_end,test_year,best_H,best_K,best_a,best_b,train_logloss,test_logloss,train_brier,test_brier
0,2015,2019,2020,25,30,0.45,1.0,0.635684,0.590036,0.222001,0.199342
1,2015,2020,2021,30,25,0.50,0.9,0.624223,0.606307,0.217288,0.210808
2,2015,2021,2022,30,15,0.55,1.0,0.621856,0.610734,0.216431,0.211762
3,2015,2022,2023,25,20,0.45,1.0,0.621385,0.600106,0.216292,0.206366
4,2015,2023,2024,35,35,0.45,0.8,0.617760,0.595180,0.214626,0.202870


In [7]:
initial_train_years = [2015, 2016, 2017, 2018, 2019]
all_years = list(range(2015, 2025))

H_list = [25, 30, 35, 40]
K_list = [15, 20, 25, 30, 35]
a_list = [0.45, 0.50, 0.55, 0.60, 0.65]
b_list = [0.8, 0.9, 1.0]

# MOV choices:
# - no MOV  -> use_mov=False (b ignored, but we'll include b=0.8 as a placeholder)
# - MOV w/ exponent -> use_mov=True and b in {0.6,0.8,1.0}

all_step_results = []   # stores full res df for each test_year
summary_rows = []

train = initial_train_years.copy()

while True:
    last_train = train[-1]
    idx = all_years.index(last_train)
    if idx + 1 >= len(all_years):
        break
    test_year = all_years[idx + 1]

    print(f"\n=== WALKFORWARD: train={train[0]}-{train[-1]} test={test_year} ===")

    res = run_gridsearch(
        train_years=train,
        test_years=[test_year],
        H_list=H_list,
        K_list=K_list,
        a_list=a_list,
        b_list=b_list,
        verbose=True,
    )

    # tag which held-out year this result corresponds to
    res = res.copy()
    res["heldout_year"] = test_year
    all_step_results.append(res)

    best = res.iloc[0].to_dict()
    summary_rows.append({
        "train_start": train[0],
        "train_end": train[-1],
        "test_year": test_year,
        "best_H": best["H"],
        "best_K": best["K"],
        "best_a": best["a"],
        "best_b": best["b"],
        "train_logloss": best["train_logloss"],
        "test_logloss": best["test_logloss"],
        "train_brier": best["train_brier"],
        "test_brier": best["test_brier"],
    })

    train = train + [test_year]

summary = pd.DataFrame(summary_rows)
summary


=== WALKFORWARD: train=2015-2019 test=2020 ===
H=25 K=15 a=0.45 b=0.8 | train_logloss=0.635897 train_brier=0.222496 | test_logloss=0.618147 test_brier=0.213476
H=25 K=15 a=0.45 b=0.9 | train_logloss=0.631629 train_brier=0.220693 | test_logloss=0.608730 test_brier=0.209099
H=25 K=15 a=0.45 b=1 | train_logloss=0.629197 train_brier=0.219708 | test_logloss=0.600398 test_brier=0.205203
H=25 K=15 a=0.5 b=0.8 | train_logloss=0.635468 train_brier=0.222324 | test_logloss=0.616353 test_brier=0.212644
H=25 K=15 a=0.5 b=0.9 | train_logloss=0.631352 train_brier=0.220593 | test_logloss=0.607476 test_brier=0.208519
H=25 K=15 a=0.5 b=1 | train_logloss=0.629102 train_brier=0.219682 | test_logloss=0.599776 test_brier=0.204899
H=25 K=15 a=0.55 b=0.8 | train_logloss=0.635152 train_brier=0.222210 | test_logloss=0.614604 test_brier=0.211839
H=25 K=15 a=0.55 b=0.9 | train_logloss=0.631211 train_brier=0.220558 | test_logloss=0.606318 test_brier=0.207983
H=25 K=15 a=0.55 b=1 | train_logloss=0.629165 train_bri

,train_start,train_end,test_year,best_H,best_K,best_a,best_b,train_logloss,test_logloss,train_brier,test_brier
0,2015,2019,2020,25,30,0.45,1.0,0.635684,0.590036,0.222001,0.199342
1,2015,2020,2021,30,25,0.50,0.9,0.624223,0.606307,0.217288,0.210808
2,2015,2021,2022,30,15,0.55,1.0,0.621856,0.610734,0.216431,0.211762
3,2015,2022,2023,25,20,0.45,1.0,0.621385,0.600106,0.216292,0.206366
4,2015,2023,2024,35,35,0.45,0.8,0.617760,0.595180,0.214626,0.202870


In [8]:
all_res = pd.concat(all_step_results, ignore_index=True)

# Mean test_logloss across heldout years for each hyperparameter tuple
tuple_means = (
    all_res
    .groupby(["H", "K", "a", "b"], as_index=False)
    .agg(
        mean_test_logloss=("test_logloss", "mean"),
        mean_test_brier=("test_brier", "mean"),
        n_steps=("heldout_year", "nunique"),
    )
    .sort_values(["mean_test_logloss", "mean_test_brier"], ascending=True)
    .reset_index(drop=True)
)

tuple_means.head(30)

,H,K,a,b,mean_test_logloss,mean_test_brier,n_steps
0,25,20,0.45,1.0,0.602076,0.207161,5
1,30,20,0.45,1.0,0.602194,0.207199,5
2,25,30,0.45,0.9,0.602319,0.207204,5
3,25,20,0.50,1.0,0.602377,0.207286,5
4,30,30,0.45,0.9,0.602422,0.207233,5
5,35,20,0.45,1.0,0.602490,0.207314,5
6,30,20,0.50,1.0,0.602493,0.207319,5
7,25,25,0.45,0.9,0.602500,0.207366,5
8,30,25,0.45,0.9,0.602628,0.207413,5
9,25,25,0.50,0.9,0.602632,0.207427,5


In [9]:
initial_train_years = [2015, 2016, 2017, 2018, 2019]
all_years = list(range(2015, 2025))

H_list = [25, 30, 35]
K_list = [15, 20, 25]
a_list = [0.425, 0.45, 0.475, 0.50]
b_list = [0.9, 0.95, 1.0, 1.05]

# MOV choices:
# - no MOV  -> use_mov=False (b ignored, but we'll include b=0.8 as a placeholder)
# - MOV w/ exponent -> use_mov=True and b in {0.6,0.8,1.0}

all_step_results = []   # stores full res df for each test_year
summary_rows = []

train = initial_train_years.copy()

while True:
    last_train = train[-1]
    idx = all_years.index(last_train)
    if idx + 1 >= len(all_years):
        break
    test_year = all_years[idx + 1]

    print(f"\n=== WALKFORWARD: train={train[0]}-{train[-1]} test={test_year} ===")

    res = run_gridsearch(
        train_years=train,
        test_years=[test_year],
        H_list=H_list,
        K_list=K_list,
        a_list=a_list,
        b_list=b_list,
        verbose=True,
    )

    # tag which held-out year this result corresponds to
    res = res.copy()
    res["heldout_year"] = test_year
    all_step_results.append(res)

    best = res.iloc[0].to_dict()
    summary_rows.append({
        "train_start": train[0],
        "train_end": train[-1],
        "test_year": test_year,
        "best_H": best["H"],
        "best_K": best["K"],
        "best_a": best["a"],
        "best_b": best["b"],
        "train_logloss": best["train_logloss"],
        "test_logloss": best["test_logloss"],
        "train_brier": best["train_brier"],
        "test_brier": best["test_brier"],
    })

    train = train + [test_year]

summary = pd.DataFrame(summary_rows)
summary


=== WALKFORWARD: train=2015-2019 test=2020 ===
H=25 K=15 a=0.425 b=0.9 | train_logloss=0.631815 train_brier=0.220766 | test_logloss=0.609391 test_brier=0.209405
H=25 K=15 a=0.425 b=0.95 | train_logloss=0.630274 train_brier=0.220137 | test_logloss=0.604906 test_brier=0.207321
H=25 K=15 a=0.425 b=1 | train_logloss=0.629299 train_brier=0.219747 | test_logloss=0.600763 test_brier=0.205378
H=25 K=15 a=0.425 b=1.05 | train_logloss=0.628996 train_brier=0.219626 | test_logloss=0.597070 test_brier=0.203606
H=25 K=15 a=0.45 b=0.9 | train_logloss=0.631629 train_brier=0.220693 | test_logloss=0.608730 test_brier=0.209099
H=25 K=15 a=0.45 b=0.95 | train_logloss=0.630129 train_brier=0.220081 | test_logloss=0.604389 test_brier=0.207080
H=25 K=15 a=0.45 b=1 | train_logloss=0.629197 train_brier=0.219708 | test_logloss=0.600398 test_brier=0.205203
H=25 K=15 a=0.45 b=1.05 | train_logloss=0.628943 train_brier=0.219605 | test_logloss=0.596860 test_brier=0.203496
H=25 K=15 a=0.475 b=0.9 | train_logloss=0.63

,train_start,train_end,test_year,best_H,best_K,best_a,best_b,train_logloss,test_logloss,train_brier,test_brier
0,2015,2019,2020,25,25,0.425,1.05,0.634830,0.589916,0.221704,0.199429
1,2015,2020,2021,30,25,0.500,0.90,0.624223,0.606307,0.217288,0.210808
2,2015,2021,2022,30,15,0.500,1.00,0.621901,0.610980,0.216432,0.211836
3,2015,2022,2023,25,20,0.425,1.00,0.621440,0.599913,0.216315,0.206276
4,2015,2023,2024,35,25,0.425,0.95,0.617582,0.595169,0.214518,0.202694


In [10]:
all_res = pd.concat(all_step_results, ignore_index=True)

# Mean test_logloss across heldout years for each hyperparameter tuple
tuple_means = (
    all_res
    .groupby(["H", "K", "a", "b"], as_index=False)
    .agg(
        mean_test_logloss=("test_logloss", "mean"),
        mean_test_brier=("test_brier", "mean"),
        n_steps=("heldout_year", "nunique"),
    )
    .sort_values(["mean_test_logloss", "mean_test_brier"], ascending=True)
    .reset_index(drop=True)
)

tuple_means.head(30)

,H,K,a,b,mean_test_logloss,mean_test_brier,n_steps
0,25,20,0.425,1.00,0.602027,0.207139,5
1,25,20,0.450,1.00,0.602076,0.207161,5
2,25,25,0.425,0.95,0.602081,0.207127,5
3,25,20,0.425,1.05,0.602099,0.207072,5
4,30,20,0.425,1.00,0.602143,0.207178,5
5,25,25,0.450,0.95,0.602153,0.207155,5
6,30,25,0.425,0.95,0.602185,0.207158,5
7,30,20,0.425,1.05,0.602192,0.207092,5
8,25,20,0.475,1.00,0.602193,0.207210,5
9,30,20,0.450,1.00,0.602194,0.207199,5
